<a href="https://colab.research.google.com/github/rwalden-gif/IntroToML/blob/main/Homework3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import SGDClassifier
from sklearn.metrics import log_loss, accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, ConfusionMatrixDisplay

data = pd.read_csv("diabetes.csv")

X = data.drop(columns=["Outcome"]).values
Y = data["Outcome"].values
m = len(Y)

print(data.head())
print("Number of samples:", m)

FileNotFoundError: [Errno 2] No such file or directory: 'diabetes.csv'

In [ ]:
def train_logistic(X_train, y_train, X_test, y_test, epochs, penalty=None, alpha=0.0001):
    model = SGDClassifier(loss="log_loss", penalty=penalty, alpha=alpha, random_state=42)
    train_loss, test_loss, train_acc, test_acc = [], [], [], []

    for i in range(epochs):
        model.partial_fit(X_train, y_train, classes=[0, 1])
        train_loss.append(log_loss(y_train, model.predict_proba(X_train)[:, 1]))
        test_loss.append(log_loss(y_test, model.predict_proba(X_test)[:, 1]))
        train_acc.append(model.score(X_train, y_train))
        test_acc.append(model.score(X_test, y_test))

    return model, train_loss, test_loss, train_acc, test_acc

def show_results(model, X_test, y_test, labels, name):
    pred = model.predict(X_test)
    print(f"--- {name} ---")
    print(f"Accuracy:  {accuracy_score(y_test, pred):.4f}")
    print(f"Precision: {precision_score(y_test, pred):.4f}")
    print(f"Recall:    {recall_score(y_test, pred):.4f}")
    print(f"F1 score:  {f1_score(y_test, pred):.4f}")

    cm = confusion_matrix(y_test, pred)
    fig, ax = plt.subplots(figsize=(6,4.5))
    ConfusionMatrixDisplay(cm, display_labels=labels).plot(ax=ax, cmap="Blues")
    plt.title(f"Confusion Matrix for {name}")
    plt.show()


In [ ]:
X_train, X_test, Y_train, Y_test = train_test_split(X, Y, test_size=0.2, random_state=42, stratify=Y)

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

print("Training samples:", len(Y_train))
print("Test samples:", len(Y_test))


In [ ]:
epochs = 150
results_p1 = {}

model, train_loss, test_loss, train_acc, test_acc = train_logistic(X_train, Y_train, X_test, Y_test, epochs)
results_p1["Diabetes"] = (model, train_loss, test_loss, train_acc, test_acc)

print(f"Final training loss: {train_loss[-1]:.5f}")
print(f"Final test loss: {test_loss[-1]:.5f}")
print(f"Final training accuracy: {train_acc[-1]:.4f}")
print(f"Final test accuracy: {test_acc[-1]:.4f}")


In [ ]:
for name, (model, train_loss, test_loss, train_acc, test_acc) in results_p1.items():

    plt.figure(figsize=(6,4.5))
    plt.plot(train_loss, color="blue", label="Training loss")
    plt.plot(test_loss, color="red", label="Test loss")
    plt.xlabel("Epoch")
    plt.ylabel("Loss")
    plt.title(f"Loss Curve for {name}")
    plt.legend()
    plt.show()

    plt.figure(figsize=(6,4.5))
    plt.plot(train_acc, color="blue", label="Training accuracy")
    plt.plot(test_acc, color="red", label="Test accuracy")
    plt.xlabel("Epoch")
    plt.ylabel("Accuracy")
    plt.title(f"Accuracy Curve for {name}")
    plt.legend()
    plt.show()


In [ ]:
for name, (model, train_loss, test_loss, train_acc, test_acc) in results_p1.items():
    show_results(model, X_test, Y_test, ["No Diabetes", "Diabetes"], name)

In [ ]:
cancer = pd.read_csv("Cancer.csv")
cancer = cancer.drop(columns=["Unnamed: 32", "id"])

Xc = cancer.drop(columns=["diagnosis"]).values
Yc = (cancer["diagnosis"] == "M").astype(int).values

print(cancer.head())
print("Number of samples:", len(Yc))
print("Number of features:", Xc.shape[1])

Xc_train, Xc_test, Yc_train, Yc_test = train_test_split(Xc, Yc, test_size=0.2, random_state=42, stratify=Yc)

scaler_c = StandardScaler()
Xc_train = scaler_c.fit_transform(Xc_train)
Xc_test = scaler_c.transform(Xc_test)

print("Training samples:", len(Yc_train))
print("Test samples:", len(Yc_test))


In [ ]:
results_p2 = {}

model, train_loss, test_loss, train_acc, test_acc = train_logistic(Xc_train, Yc_train, Xc_test, Yc_test, epochs)
results_p2["No Penalty"] = (model, train_loss, test_loss, train_acc, test_acc)

print(f"Final training loss: {train_loss[-1]:.5f}")
print(f"Final test loss: {test_loss[-1]:.5f}")
print(f"Final training accuracy: {train_acc[-1]:.4f}")
print(f"Final test accuracy: {test_acc[-1]:.4f}")


In [ ]:
penalty_strengths = [0.0001, 0.001, 0.01, 0.1]
best_alpha, best_result = None, None

for alpha in penalty_strengths:
    model, train_loss, test_loss, train_acc, test_acc = train_logistic(Xc_train, Yc_train, Xc_test, Yc_test, epochs, penalty="l2", alpha=alpha)
    print(f"alpha={alpha} -> final test loss={test_loss[-1]:.5f}, final test accuracy={test_acc[-1]:.4f}")

    if best_result is None or test_loss[-1] < best_result[2][-1]:
        best_alpha, best_result = alpha, (model, train_loss, test_loss, train_acc, test_acc)

results_p2["With Penalty"] = best_result
print(f">> Best penalty strength: alpha={best_alpha}")


In [ ]:
for name, (model, train_loss, test_loss, train_acc, test_acc) in results_p2.items():

    plt.figure(figsize=(6,4.5))
    plt.plot(train_loss, color="blue", label="Training loss")
    plt.plot(test_loss, color="red", label="Test loss")
    plt.xlabel("Epoch")
    plt.ylabel("Loss")
    plt.title(f"Loss Curve for Cancer, {name}")
    plt.legend()
    plt.show()

    plt.figure(figsize=(6,4.5))
    plt.plot(train_acc, color="blue", label="Training accuracy")
    plt.plot(test_acc, color="red", label="Test accuracy")
    plt.xlabel("Epoch")
    plt.ylabel("Accuracy")
    plt.title(f"Accuracy Curve for Cancer, {name}")
    plt.legend()
    plt.show()


In [ ]:
for name, (model, train_loss, test_loss, train_acc, test_acc) in results_p2.items():
    show_results(model, Xc_test, Yc_test, ["Benign", "Malignant"], "Cancer, " + name)

final_f1 = {}
final_f1["Diabetes"] = f1_score(Y_test, results_p1["Diabetes"][0].predict(X_test))
final_f1["Cancer, No Penalty"] = f1_score(Yc_test, results_p2["No Penalty"][0].predict(Xc_test))
final_f1["Cancer, With Penalty"] = f1_score(Yc_test, results_p2["With Penalty"][0].predict(Xc_test))

for k, v in final_f1.items():
    print(f"{k}: F1 score = {v:.4f}")

plt.figure(figsize=(6,4.5))
plt.bar(final_f1.keys(), final_f1.values(), color=["#4C72B0","#DD8452","#55A868"])
plt.ylabel("F1 Score")
plt.title("Test F1 Score by Model")
for i, (k, v) in enumerate(final_f1.items()):
    plt.text(i, v+0.01, f"{v:.3f}", ha="center")
plt.show()
